In [60]:
from datasets import load_dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.naive_bayes import MultinomialNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

# Task 1

In [2]:
langs={'af-ZA', 'da-DK', 'de-DE', 'en-US', 'es-ES', 'fr-FR', 'fi-FI', 'hu-HU', 'is-IS', 'it-IT', 
 'jv-ID', 'lv-LV', 'ms-MY', 'nb-NO', 'nl-NL', 'pl-PL', 'pt-PT', 'ro-RO', 'ru-RU', 'sl-SL', 
 'sv-SE', 'sq-AL', 'sw-KE', 'tl-PH', 'tr-TR', 'vi-VN', 'cy-GB'}

In [3]:
# train_data=load_dataset('qanastek/MASSIVE', split='train')
# train_df=pd.DataFrame({'locale': train_data['locale'],
#                        'utt': train_data['utt']})
# train_df=train_df.loc[train_df['locale'].isin(langs), :]
# train_df.to_csv('train_data.tsv', sep='\t', encoding='utf-16', index=False)

In [4]:
# del train_data
# test_data=load_dataset('qanastek/MASSIVE', split='test')
# test_df=pd.DataFrame({'locale': test_data['locale'],
#                        'utt': test_data['utt']})
# test_df=test_df.loc[test_df['locale'].isin(langs), :]
# test_df.to_csv('test_data.tsv', sep='\t', encoding='utf-16', index=False)

In [5]:
# del test_data
# validation_data=load_dataset('qanastek/MASSIVE', split='validation')
# validation_df=pd.DataFrame({'locale': validation_data['locale'],
#                        'utt': validation_data['utt']})
# validation_df=validation_df.loc[validation_df['locale'].isin(langs), :]
# validation_df.to_csv('validation_data.tsv', sep='\t', encoding='utf-16', index=False)

# Task 2

In [3]:
train_df=pd.read_csv('train_data.tsv', sep='\t', encoding='utf-16')
test_df=pd.read_csv('test_data.tsv', sep='\t', encoding='utf-16')
validation_df=pd.read_csv('validation_data.tsv', sep='\t', encoding='utf-16')

In [4]:
X_train, y_train=train_df['utt'], train_df['locale']
X_test, y_test=test_df['utt'], test_df['locale']
X_validation, y_validation=validation_df['utt'], validation_df['locale']

In [5]:
pipeline=Pipeline(steps=[('vectorizer', TfidfVectorizer(encoding='utf-16')),
                         ('clsf', MultinomialNB())])

In [6]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('vectorizer', TfidfVectorizer(encoding='utf-16')),
                ('clsf', MultinomialNB())])

In [7]:
X=pd.concat((X_train, X_validation), axis=0, ignore_index=True)
y=pd.concat((y_train, y_validation), ignore_index=True)
fold=np.zeros(X.shape[0])
fold[:X_train.shape[0]]=-1

In [8]:
ps=PredefinedSplit(test_fold=fold)
gscv=GridSearchCV(estimator=pipeline,
                  param_grid={'vectorizer__max_features': list(range(130000, 180000, 10000)),
                              'clsf__alpha': np.linspace(0.01, 0.5, 10)},
                  cv=ps, n_jobs=-1)
gscv.fit(X, y)

GridSearchCV(cv=PredefinedSplit(test_fold=array([-1, -1, ...,  0,  0])),
             estimator=Pipeline(steps=[('vectorizer',
                                        TfidfVectorizer(encoding='utf-16')),
                                       ('clsf', MultinomialNB())]),
             n_jobs=-1,
             param_grid={'clsf__alpha': array([0.01      , 0.06444444, 0.11888889, 0.17333333, 0.22777778,
       0.28222222, 0.33666667, 0.39111111, 0.44555556, 0.5       ]),
                         'vectorizer__max_features': [130000, 140000, 150000,
                                                      160000, 170000]})

In [9]:
gscv.best_params_, gscv.best_score_

({'clsf__alpha': np.float64(0.11888888888888888),
  'vectorizer__max_features': 160000},
 np.float64(0.9850430853874041))

In [10]:
def model_report(model):
    train_predictions=model.predict(X_train)
    test_predictions=model.predict(X_test)
    validation_predictions=model.predict(X_validation)

    print('Performance on Training Data:')
    print('\tAccuracy: %.3f'%(accuracy_score(y_train, train_predictions)*100))
    print('\tPrecision: %.3f'%(precision_score(y_train, train_predictions, average='macro')*100))
    print('\tRecall: %.3f'%(recall_score(y_train, train_predictions, average='macro')*100))
    print('\tF1: %.3f'%(f1_score(y_train, train_predictions, average='macro')*100), end='\n\n')

    print('Performance on Validation Data:')
    print('\tAccuracy: %.3f'%(accuracy_score(y_validation, validation_predictions)*100))
    print('\tPrecision: %.3f'%(precision_score(y_validation, validation_predictions, average='macro')*100))
    print('\tRecall: %.3f'%(recall_score(y_validation, validation_predictions, average='macro')*100))
    print('\tF1: %.3f'%(f1_score(y_validation, validation_predictions, average='macro')*100), end='\n\n')

    print('Performance on Testing Data:')
    print('\tAccuracy: %.3f'%(accuracy_score(y_test, test_predictions)*100))
    print('\tPrecision: %.3f'%(precision_score(y_test, test_predictions, average='macro')*100))
    print('\tRecall: %.3f'%(recall_score(y_test, test_predictions, average='macro')*100))
    print('\tF1: %.3f'%(f1_score(y_test, test_predictions, average='macro')*100))

In [11]:
model_report(gscv)

Performance on Training Data:
	Accuracy: 99.295
	Precision: 99.298
	Recall: 99.295
	F1: 99.296

Performance on Validation Data:
	Accuracy: 99.191
	Precision: 99.196
	Recall: 99.191
	F1: 99.192

Performance on Testing Data:
	Accuracy: 98.537
	Precision: 98.585
	Recall: 98.537
	F1: 98.548


# Task 3

In [12]:
groups={'af-ZA': 'Africa', 'cy-GB': 'Europe', 'da-DK': 'Europe', 'de-DE': 'Europe',
        'en-US': 'North America', 'es-ES': 'Europe', 'fi-FI': 'Europe', 'fr-FR': 'Europe',
        'hu-HU': 'Europe', 'is-IS': 'Europe', 'it-IT': 'Europe', 'jv-ID': 'Asia',
        'lv-LV': 'Europe', 'ms-MY': 'Asia', 'nb-NO': 'Europe', 'nl-NL': 'Europe',
        'pl-PL': 'Europe', 'pt-PT': 'Europe', 'ro-RO': 'Europe', 'ru-RU': 'Asia',
        'sl-SL': 'Europe', 'sq-AL': 'Europe', 'sv-SE': 'Europe', 'sw-KE': 'Africa',
        'tl-PH': 'Asia', 'tr-TR': 'Asia', 'vi-VN': 'Asia'}

y_train=y_train.apply(lambda x: groups[x])
y_test=y_test.apply(lambda x: groups[x])
y_validation=y_validation.apply(lambda x: groups[x])

In [20]:
vec=TfidfVectorizer(encoding='utf-16', max_features=5000)
z=vec.fit_transform(X_train)
z.toarray().shape

(310878, 5000)

In [58]:
trans=Pipeline(steps=[('vectorizer', TfidfVectorizer(encoding='utf-16', max_features=10000)),
                    #   ('dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
                      ('pca', PCA(n_components=500))])
z=trans.fit_transform(X_train, y_train)
z.shape

(310878, 500)

In [61]:
class LDA_QDA(BaseEstimator, ClassifierMixin):
    def __init__(self, qda_ratio=0):
        self.qda_ratio=qda_ratio
        self.lda=LinearDiscriminantAnalysis()
        self.qda=QuadraticDiscriminantAnalysis()
        self._is_fitted=False
    
    def fit(self, X, y):
        if self.qda_ratio==0:
            self.lda.fit(X, y)
        elif self.qda_ratio==1:
            self.qda.fit(X, y)
        else:
            self.lda.fit(X, y)
            self.qda.fit(X, y)
        self._is_fitted=True
        return self
    
    def predict(self, X):
        if self.qda_ratio==0:
            return self.lda.predict(X)
        elif self.qda_ratio==1:
            return self.qda.predict(X)
        else:
            lda_proba=self.lda.predict_proba(X)
            qda_proba=self.qda.predict_proba(X)
            proba=(1-self.qda_ratio)*lda_proba + self.qda_ratio*qda_proba
            return  self.lda.classes_[proba.argmax(axis=1)]

    def __sklearn_is_fitted__(self):
        return self._is_fitted

In [62]:
clsf3=LDA_QDA(0)
clsf3.fit(z, y_train)

LDA_QDA()

In [65]:
clsf3.lda.score(z, y_train)

0.9356757313158216

In [66]:
clsf4=LDA_QDA(1)
clsf4.fit(z, y_train)

LDA_QDA(qda_ratio=1)

In [67]:
clsf4.qda.score(z, y_train)

0.9538307631932784

In [68]:
clsf5=LDA_QDA(0.5)
clsf5.fit(z, y_train)

LDA_QDA(qda_ratio=0.5)

In [69]:
predictions=clsf5.predict(z)
print((y_train==predictions).mean())

0.9540237649495944
